In [ ]:
# =============================================================================
# 0. INSTALL DEPENDENCIES (Kaggle)
# =============================================================================
!apt-get update -qq
!apt-get install -y -qq openbabel autodock-vina
!pip install -q biotite meeko openmm rdkit scipy seaborn matplotlib gemmi

print('=== Dependency installation complete ===')

import importlib
for pkg in ['biotite', 'meeko', 'openmm', 'rdkit', 'scipy', 'seaborn', 'matplotlib', 'gemmi']:
    importlib.import_module(pkg)
    print(f'  {pkg}: OK')

In [ ]:
# =============================================================================
# 1. PATH SETUP FOR KAGGLE
# =============================================================================
import os
WORKING_DIR = '/kaggle/working'
DATA_DIR = os.path.join(WORKING_DIR, 'data')
RESULTS_DIR = os.path.join(WORKING_DIR, 'results')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'WORKING_DIR: {WORKING_DIR}')
print(f'DATA_DIR:    {DATA_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')

In [ ]:
# =============================================================================
# 2. GPR151 CONTEXTUALIZATION DASHBOARD
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
import biotite.structure.io as strucio
import biotite.structure as struc
import biotite.database.afdb as afdb

print('Downloading GPR151 (AlphaFold ID: Q8TDV0)...')
cif_filename = afdb.fetch('Q8TDV0', 'cif', target_path=DATA_DIR)
array = strucio.load_structure(cif_filename)
protein = array[struc.filter_amino_acids(array)]
res_ids = np.unique(protein.res_id)
calphas = protein[protein.atom_name == 'CA']

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('GPR151 Contextualization Dashboard', fontsize=18)

mock_entropy = np.random.uniform(0.1, 2.5, size=len(res_ids))
axs[0, 0].plot(res_ids, mock_entropy, color='purple', alpha=0.7)
axs[0, 0].set_title('1. Sequence Conservation (Mock Entropy)')
axs[0, 0].set_xlabel('Residue Number')
axs[0, 0].set_ylabel('Shannon Entropy (Bits)')

print('Calculating SASA...')
atom_sasa = struc.sasa(protein)
res_sasa = np.array([np.sum(atom_sasa[protein.res_id == r]) for r in res_ids])
axs[0, 1].plot(res_ids, res_sasa, color='teal')
axs[0, 1].set_title('2. Per-Residue SASA')
axs[0, 1].set_xlabel('Residue Number')
axs[0, 1].set_ylabel('SASA (Å²)')

print('Generating Distance Matrix...')
coords = calphas.coord
dist_matrix = cdist(coords, coords, metric='euclidean')
sns.heatmap(dist_matrix, ax=axs[1, 0], cmap='viridis_r',
            cbar_kws={'label': 'Distance (Å)'},
            xticklabels=50, yticklabels=50)
axs[1, 0].set_title('3. C-alpha Contact Map')
axs[1, 0].set_xlabel('Residue Index')
axs[1, 0].set_ylabel('Residue Index')

mock_energy = np.random.normal(-1, 2, size=len(res_ids))
mock_energy[mock_energy > 0] = 0
axs[1, 1].bar(res_ids, mock_energy, color='crimson')
axs[1, 1].set_title('4. Interaction Energy (Mock Data)')
axs[1, 1].set_xlabel('Residue Number')
axs[1, 1].set_ylabel('Energy Contribution (kcal/mol)')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plot_path = os.path.join(RESULTS_DIR, 'GPR151_contextualization_dashboard.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f'Saved plot: {plot_path}')
plt.show()

In [ ]:
# =============================================================================
# 3. FETCH & CLEAN RECEPTOR (GPR151)
# =============================================================================
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
import biotite.structure.io as strucio
import biotite.structure as struc
import biotite.database.afdb as afdb
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation

LIGAND_SMILES = 'COC1=CC=CC=C1CNC(=O)CC2=C(C)OC3=C(C2=O)C=C4C(=C3C)OC=C4'
LIGAND_RES_NAME = 'UNL'

print('Downloading GPR151...')
cif_filename = afdb.fetch('Q8TDV0', 'cif', target_path=DATA_DIR)
protein = strucio.load_structure(cif_filename)
protein = protein[struc.filter_amino_acids(protein)]

protein_clean_path = os.path.join(RESULTS_DIR, 'GPR151_clean.pdb')
strucio.save_structure(protein_clean_path, protein)
print(f'Saved cleaned protein: {protein_clean_path} ({len(np.unique(protein.res_id))} residues)')

In [ ]:
# =============================================================================
# 4. RECEPTOR PREP (NO OPENMM - FAST)
# =============================================================================
# The previous OpenMM solvation step created a 40MB+ file with thousands of
# water molecules, causing Open Babel to hang for 10+ hours.
# For docking, we only need the protein with hydrogens.

prepared_pdbqt = os.path.join(RESULTS_DIR, 'GPR151_prepared.pdbqt')

print('Converting clean protein to PDBQT with Open Babel...')
cmd = f'obabel {protein_clean_path} -O {prepared_pdbqt} -xr -p 7.4'
print(f'Running: {cmd}')
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError('Open Babel failed.')
if not os.path.exists(prepared_pdbqt) or os.path.getsize(prepared_pdbqt) == 0:
    raise RuntimeError('Receptor PDBQT is empty.')
print(f'SUCCESS: {prepared_pdbqt} ({os.path.getsize(prepared_pdbqt)} bytes)')

# Optional quick preview
print(f'Clean PDB size:   {os.path.getsize(protein_clean_path):,} bytes')
print(f'Prepared PDBQT:   {os.path.getsize(prepared_pdbqt):,} bytes')

In [ ]:
# =============================================================================
# 5. LIGAND PREPARATION (GUM3)
# =============================================================================
print('Preparing Ligand (GUM3)...')
mol = Chem.MolFromSmiles(LIGAND_SMILES)
if mol is None:
    raise ValueError(f'Failed to parse SMILES: {LIGAND_SMILES}')
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)

prep = MoleculePreparation()
prep.prepare(mol)
ligand_pdbqt = os.path.join(RESULTS_DIR, 'GUM3.pdbqt')
with open(ligand_pdbqt, 'w') as f:
    f.write(prep.write_pdbqt_string())
if not os.path.exists(ligand_pdbqt) or os.path.getsize(ligand_pdbqt) == 0:
    raise RuntimeError('Ligand PDBQT is empty.')
print(f'SUCCESS: {ligand_pdbqt} ({os.path.getsize(ligand_pdbqt)} bytes)')

In [ ]:
# =============================================================================
# 6. MOLECULAR DOCKING WITH AUTODOCK VINA
# =============================================================================
docked_pdbqt = os.path.join(RESULTS_DIR, 'GUM3_docked.pdbqt')
vina_log = os.path.join(RESULTS_DIR, 'vina_log.txt')

print('Running Vina Docking...')
cmd = [
    'vina',
    '--receptor', prepared_pdbqt,
    '--ligand', ligand_pdbqt,
    '--center_x', '-5.350',
    '--center_y', '2.548',
    '--center_z', '1.510',
    '--size_x', '15',
    '--size_y', '15',
    '--size_z', '15',
    '--exhaustiveness', '8',
    '--out', docked_pdbqt
]
result = subprocess.run(cmd, capture_output=True, text=True)
with open(vina_log, 'w') as f:
    f.write(result.stdout)
    f.write(result.stderr)
print(result.stdout)
if result.returncode != 0:
    print(f'ERROR: Vina exited {result.returncode}')
    print(result.stderr)
else:
    print(f'\nSUCCESS: Docking complete -> {docked_pdbqt}')
    print(f'Log saved -> {vina_log}')
    print('\n--- Top poses ---')
    for line in result.stdout.splitlines():
        if line.strip().startswith(('1 ', '2 ', '3 ')) and 'kcal/mol' not in line:
            print(line.strip())
if not os.path.exists(docked_pdbqt) or os.path.getsize(docked_pdbqt) == 0:
    raise RuntimeError('Docked output is empty. Check Vina log.')

In [ ]:
# =============================================================================
# 7. POST-DOCKING ANALYSIS & VISUALIZATION
# =============================================================================
print('Loading results...')
protein = strucio.load_structure(protein_clean_path)
ligand = strucio.load_structure(docked_pdbqt)
if hasattr(ligand, 'stack_depth') and ligand.stack_depth() > 1:
    ligand_first = ligand[0]
else:
    ligand_first = ligand
lig_coords = ligand_first.coord
res_ids = np.unique(protein.res_id)

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('GPR151 + GUM3 Docking Analysis', fontsize=18)

axs[0, 0].plot(res_ids, np.random.uniform(0.1, 2.5, size=len(res_ids)), color='purple')
axs[0, 0].set_title('1. Sequence Conservation (Mock)')
axs[0, 0].set_xlabel('Residue ID')
axs[0, 0].set_ylabel('Entropy Score')

atom_sasa = struc.sasa(protein)
res_sasa = np.array([np.sum(atom_sasa[protein.res_id == r]) for r in res_ids])
axs[0, 1].plot(res_ids, res_sasa, color='teal')
axs[0, 1].set_title('2. Solvent Accessible Surface Area')
axs[0, 1].set_xlabel('Residue ID')
axs[0, 1].set_ylabel(r'SASA ($\AA^2$)')

unique_res_ids = np.unique(protein.res_id)
contact_matrix = np.zeros((len(unique_res_ids), len(lig_coords)))
for i, res_id in enumerate(unique_res_ids):
    res_atoms = protein[protein.res_id == res_id]
    dists = cdist(res_atoms.coord, lig_coords)
    contact_matrix[i, :] = np.min(dists, axis=0)
sns.heatmap(contact_matrix, ax=axs[1, 0], cmap='rocket_r', yticklabels=40)
axs[1, 0].set_title('3. Ligand-Protein Contact Distances')
axs[1, 0].set_xlabel('Ligand Atom Index')
axs[1, 0].set_ylabel('Protein Residue ID')

dist_to_lig = cdist(protein.coord, lig_coords).min(axis=1)
contact_counts = np.array([np.sum(dist_to_lig[protein.res_id == r] < 4.0) for r in res_ids])
axs[1, 1].bar(res_ids, contact_counts, color='orange')
axs[1, 1].set_title('4. Interaction Proxy (Contacts < 4Å)')
axs[1, 1].set_xlabel('Residue ID')
axs[1, 1].set_ylabel('Number of Contacts')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plot_path = os.path.join(RESULTS_DIR, 'GPR151_GUM3_dashboard.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f'Saved plot: {plot_path}')
plt.show()

print('\n=== ALL DONE ===')
print(f'Outputs in: {RESULTS_DIR}')
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f'  - {f}')